# Unit 4: CNN 训练实战

## 学习目标
- 深入理解 `Dataset` 和 `DataLoader` 的工作机制
- 掌握数据预处理和增强 (transforms)
- 编写健壮、可复用的训练/验证循环
- 学会保存和加载模型
- 可视化特征图和训练过程
- 在 CIFAR-10 上训练一个 CNN

## 4.1 Dataset 和 DataLoader

```
Dataset:     定义如何读取单个样本 (__getitem__)
DataLoader:  批量加载、打乱、多线程、pin_memory
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from tqdm import tqdm

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

## 4.2 自定义 Dataset

继承 `torch.utils.data.Dataset`，必须实现：
- `__len__`: 返回数据集大小
- `__getitem__`: 返回第 idx 个样本

In [ ]:
class SyntheticImageDataset(Dataset):
    def __init__(self, num_samples=1000, num_classes=10, transform=None):
        self.num_samples = num_samples
        self.num_classes = num_classes
        self.transform = transform
        self.data = torch.randn(num_samples, 3, 32, 32)
        self.labels = torch.randint(0, num_classes, (num_samples,))

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        img = self.data[idx]
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

dataset = SyntheticImageDataset(num_samples=100)
print(f"Dataset size: {len(dataset)}")
img, label = dataset[0]
print(f"Sample shape: {img.shape}, label: {label}")

### DataLoader 核心参数

| 参数 | 作用 | 建议值 |
|------|------|--------|
| `batch_size` | 每批样本数 | 32/64/128 (GPU 内存允许越大越好) |
| `shuffle` | 是否打乱 | Train=True, Val/Test=False |
| `num_workers` | 加载数据的子进程数 | 0=主进程, 4-8 可加速 |
| `pin_memory` | 锁页内存加速 GPU 传输 | GPU 训练时设为 True |
| `drop_last` | 丢弃最后不完整的 batch | 配合 BatchNorm 时建议 True |

In [ ]:
loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

batch = next(iter(loader))
data, labels = batch
print(f"Batch data: {data.shape}")
print(f"Batch labels: {labels.shape}")
print(f"Number of batches: {len(loader)}")

## 4.3 Transforms：数据预处理与增强

`transforms.Compose` 将多个变换串联起来。

训练时的典型流程：
```python
transforms.Compose([
    RandomHorizontalFlip(),   # 随机水平翻转
    RandomCrop(32, padding=4), # 随机裁剪 (padding 后)
    ColorJitter(...),          # 颜色抖动
    ToTensor(),               # 转为 Tensor [0,1]
    Normalize(mean, std),     # 标准化
])
```

In [ ]:
cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)

print(f"Train: {len(train_dataset):,}, Test: {len(test_dataset):,}")
print(f"Classes: {train_dataset.classes}")

In [ ]:
def denormalize(img_tensor, mean, std):
    img = img_tensor.clone()
    for t, m, s in zip(img, mean, std):
        t.mul_(s).add_(m)
    return img.clamp(0, 1)

raw_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transforms.ToTensor())
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(10):
    img, label = raw_dataset[i]
    ax = axes[i // 5][i % 5]
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(f"{train_dataset.classes[label]}")
    ax.axis("off")
plt.suptitle("CIFAR-10 Samples")
plt.tight_layout()
plt.show()

### 数据增强效果可视化

In [ ]:
demo_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
])

demo_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=None)
original_img = demo_dataset[0][0]

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
axes[0].imshow(original_img)
axes[0].set_title("Original")
axes[0].axis("off")

for i in range(1, 5):
    augmented = demo_aug(original_img)
    axes[i].imshow(augmented.permute(1, 2, 0))
    axes[i].set_title(f"Augmented #{i}")
    axes[i].axis("off")
plt.suptitle("Data Augmentation Examples")
plt.tight_layout()
plt.show()

## 4.4 训练/验证集划分

In [ ]:
train_size = int(0.9 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_subset, batch_size=128, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_subset, batch_size=128, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

## 4.5 构建 CNN 模型

In [ ]:
class CIFAR10_CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.2),
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.3),
        )
        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.classifier(x)
        return x

model = CIFAR10_CNN().to(device)
print(f"参数量: {sum(p.numel() for p in model.parameters()):,}")

x = torch.randn(2, 3, 32, 32).to(device)
with torch.no_grad():
    out = model(x)
print(f"Input {x.shape} -> Output {out.shape}")

## 4.6 完善的训练脚本

In [ ]:
class Trainer:
    def __init__(self, model, device, criterion, optimizer, scheduler=None):
        self.model = model
        self.device = device
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
        self.best_val_acc = 0.0

    def train_epoch(self, loader):
        self.model.train()
        total_loss = 0
        correct = 0
        total = 0
        pbar = tqdm(loader, desc="Training", leave=False)
        for data, target in pbar:
            data, target = data.to(self.device), target.to(self.device)
            self.optimizer.zero_grad()
            output = self.model(data)
            loss = self.criterion(output, target)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item() * data.size(0)
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += data.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        return total_loss / total, correct / total

    @torch.no_grad()
    def evaluate(self, loader, desc="Evaluating"):
        self.model.eval()
        total_loss = 0
        correct = 0
        total = 0
        for data, target in tqdm(loader, desc=desc, leave=False):
            data, target = data.to(self.device), target.to(self.device)
            output = self.model(data)
            loss = self.criterion(output, target)
            total_loss += loss.item() * data.size(0)
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += data.size(0)
        return total_loss / total, correct / total

    def fit(self, train_loader, val_loader, epochs, save_path=None):
        for epoch in range(epochs):
            train_loss, train_acc = self.train_epoch(train_loader)
            val_loss, val_acc = self.evaluate(val_loader, desc="Validating")

            if self.scheduler:
                self.scheduler.step()

            self.history["train_loss"].append(train_loss)
            self.history["train_acc"].append(train_acc)
            self.history["val_loss"].append(val_loss)
            self.history["val_acc"].append(val_acc)

            lr = self.optimizer.param_groups[0]["lr"]
            print(f"Epoch {epoch+1:3d}/{epochs} | "
                  f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
                  f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
                  f"LR: {lr:.6f}")

            if val_acc > self.best_val_acc and save_path:
                self.best_val_acc = val_acc
                self.save(save_path)
                print(f"  -> Saved best model (val_acc={val_acc:.4f})")

    def save(self, path):
        torch.save({
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "best_val_acc": self.best_val_acc,
            "history": self.history,
        }, path)

    def load(self, path):
        checkpoint = torch.load(path, map_location=self.device, weights_only=False)
        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        self.best_val_acc = checkpoint["best_val_acc"]
        self.history = checkpoint["history"]
        print(f"Loaded checkpoint from {path} (best_val_acc={self.best_val_acc:.4f})")

In [ ]:
model = CIFAR10_CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

trainer = Trainer(model, device, criterion, optimizer, scheduler)

save_dir = Path("./checkpoints")
save_dir.mkdir(exist_ok=True)

trainer.fit(train_loader, val_loader, epochs=20, save_path=str(save_dir / "cifar10_cnn.pt"))

## 4.7 训练结果分析

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(trainer.history["train_loss"], "b-", label="Train")
axes[0].plot(trainer.history["val_loss"], "r-", label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss Curves")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(trainer.history["train_acc"], "b-", label="Train")
axes[1].plot(trainer.history["val_acc"], "r-", label="Val")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy Curves")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f"CIFAR10 CNN Training (Best Val Acc: {trainer.best_val_acc:.2%})", fontsize=13)
plt.tight_layout()
plt.show()

## 4.8 测试集最终评估

In [ ]:
best_checkpoint = save_dir / "cifar10_cnn.pt"
if best_checkpoint.exists():
    model = CIFAR10_CNN().to(device)
    checkpoint = torch.load(best_checkpoint, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])

test_loss, test_acc = trainer.evaluate(test_loader, desc="Testing")
print(f"\n{'='*50}")
print(f"Test Loss:  {test_loss:.4f}")
print(f"Test Acc:   {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"{'='*50}")

## 4.9 特征图可视化

观察 CNN 各层学到了什么特征。

In [ ]:
@torch.no_grad()
def visualize_feature_maps(model, layer_name, input_tensor):
    model.eval()
    features = {}

    def hook_fn(name):
        def hook(module, input, output):
            features[name] = output.detach()
        return hook

    hook_handles = []
    for name, module in model.named_modules():
        if name == layer_name:
            hook_handles.append(module.register_forward_hook(hook_fn(name)))

    _ = model(input_tensor)

    for h in hook_handles:
        h.remove()

    return features.get(layer_name, None)

sample_img, sample_label = test_dataset[0]
sample_batch = sample_img.unsqueeze(0).to(device)

fmaps = visualize_feature_maps(model, "conv_block1.0", sample_batch)
if fmaps is not None:
    fmaps = fmaps.cpu().squeeze(0)
    n = min(16, fmaps.shape[0])
    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    for i, ax in enumerate(axes.flat):
        if i < n:
            ax.imshow(fmaps[i], cmap="viridis")
        ax.axis("off")
    plt.suptitle(f"Feature Maps: conv_block1.0 ({n} of {fmaps.shape[0]} channels)")
    plt.tight_layout()
    plt.show()

## 4.10 单元小结

| 概念 | 要点 |
|------|------|
| **Dataset** | 实现 `__len__` + `__getitem__`，封装数据读取逻辑 |
| **DataLoader** | batch, shuffle, num_workers, pin_memory |
| **Transforms** | 训练：随机增强；测试：仅标准化 |
| **训练循环** | `train()` 模式，梯度更新；`eval()` 模式，no_grad |
| **Checkpoint** | 保存 model + optimizer + history |
| **特征图** | 浅层检测边缘纹理，深层检测语义 |

### 思考题
1. 为什么训练时 shuffle=True 而测试时 shuffle=False？
2. `torch.no_grad()` 在评估时的两个作用是什么？
3. 保存 checkpoint 时为什么要同时保存 optimizer 的状态？